In [133]:
from typing import Any, Callable
from sqlmodel import create_engine, select, Session
from sqlalchemy.engine import Engine
from sqlalchemy import event
from experiment import Model, Result, Celltype, Dataset, Sample, NumericArray
import pandas as pd
import mlflow
import logging
import polars as pl
import nico2_lib as n2l
import seaborn as sns
import numpy as np

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

In [2]:
exp_name = "Default"
benchmark_experiment = mlflow.get_experiment_by_name(exp_name)
if not benchmark_experiment:
    raise ValueError(f"Experiment '{exp_name}' not found.")
runs = mlflow.search_runs(experiment_ids=[benchmark_experiment.experiment_id])
if runs.empty:
    raise RuntimeError(f"No runs found in experiment '{exp_name}'.")
last_run_id = runs.sort_values("start_time", ascending=False)["run_id"].iloc[0]
logger.info(f"Using run_id: {last_run_id}")
logger.info("Downloading 'database.db'...")
database_path = mlflow.artifacts.download_artifacts(
    run_id=last_run_id, artifact_path="database.db"
)
logger.info(f"Local path: {database_path}")
sqlite_url = f"sqlite:///{database_path}"
engine = create_engine(sqlite_url, echo=True)
logger.info("Database engine initialized.")

INFO: Using run_id: e112ee433c2648e69beb412944c74bf8
INFO: Downloading 'database.db'...
INFO: Local path: /var/folders/qm/v_v5_1r52bx792m7x2mh177c0000gn/T/tmp_p4_5cqp/database.db
INFO: Database engine initialized.


In [ ]:
statement = select(Result, Celltype, Sample, Model, Dataset).where(Celltype.name == "A")

In [109]:
with Session(engine) as session:
    results = session.exec(statement).all()

2026-03-04 17:15:12,801 INFO sqlalchemy.engine.Engine BEGIN (implicit)


2026/03/04 17:15:12 INFO sqlalchemy.engine.Engine: BEGIN (implicit)


2026-03-04 17:15:12,848 INFO sqlalchemy.engine.Engine SELECT result.id, result.global_model_embedding, result.global_model_counts, result.celltype_model_embedding, result.celltype_model_counts, result.celltype_id, result.model_id, result.sample_id, celltype.id AS id_1, celltype.name, celltype.counts_matrix, celltype.pca_embedding, celltype.umap_embedding, celltype.adjacency_matrix, celltype.dataset_id, sample.id AS id_2, sample.id_of_sample, sample.train_idx, sample.test_idx, sample.dataset_id AS dataset_id_1, model.id AS id_3, model.name AS name_1, dataset.id AS id_4, dataset.name AS name_2 
FROM result, celltype, sample, model, dataset 
WHERE celltype.name = ?


/Users/egerc/Documents/Projects/notebook_repository/notebooks/2026-02-27T09-05-08Z/.venv/lib/python3.13/site-packages/sqlmodel/orm/session.py:75: SAWarning: SELECT statement has a cartesian product between FROM element(s) "dataset", "model", "result", "sample" and FROM element "celltype".  Apply join condition(s) between each element to resolve.
  results = super().execute(
2026/03/04 17:15:12 INFO sqlalchemy.engine.Engine: SELECT result.id, result.global_model_embedding, result.global_model_counts, result.celltype_model_embedding, result.celltype_model_counts, result.celltype_id, result.model_id, result.sample_id, celltype.id AS id_1, celltype.name, celltype.counts_matrix, celltype.pca_embedding, celltype.umap_embedding, celltype.adjacency_matrix, celltype.dataset_id, sample.id AS id_2, sample.id_of_sample, sample.train_idx, sample.test_idx, sample.dataset_id AS dataset_id_1, model.id AS id_3, model.name AS name_1, dataset.id AS id_4, dataset.name AS name_2 
FROM result, celltype, sam

2026-03-04 17:15:12,853 INFO sqlalchemy.engine.Engine [generated in 0.00505s] ('A',)


2026/03/04 17:15:12 INFO sqlalchemy.engine.Engine: [generated in 0.00505s] ('A',)


2026-03-04 17:15:15,278 INFO sqlalchemy.engine.Engine ROLLBACK


2026/03/04 17:15:15 INFO sqlalchemy.engine.Engine: ROLLBACK


In [115]:
MetricResult = list[dict[str, Any]]
Entry = tuple[Result, Celltype, Sample, Model, Dataset]
MetricResultFn = Callable[[Entry], MetricResult]


def construct_result_table(
    results: list[Entry],
    extract_results_fn: MetricResultFn,
) -> pl.DataFrame:

    rows: MetricResult = []

    for result, celltype, sample, model, dataset in results:
        extracted_results = extract_results_fn(
            (result, celltype, sample, model, dataset)
        )

        for extracted_result in extracted_results:
            rows.append(
                {
                    **extracted_result,
                    "dataset_name": dataset.name,
                    "celltype": celltype.name,
                    "sample_id": sample.id_of_sample,
                    "model_name": model.name,
                }
            )

    return pl.DataFrame(rows)

In [146]:
def metric_results_fn(entry: Entry) -> MetricResult:
    result, celltype, sample, _, _ = entry
    # pearson_score = n2l.mt.pearson_metric(result.celltype_model_counts, celltype.counts_matrix[:, sample.test_idx])
    rng = np.random.default_rng()
    return [
        {"metric_name": "pearson", "score": rng.normal()},
        {"metric_name": "spearman", "score": rng.normal()}
    ]

In [147]:
df = construct_result_table(results=results, extract_results_fn=metric_results_fn)

In [148]:
df

metric_name,score,dataset_name,celltype,sample_id,model_name
str,f64,str,str,i64,str
"""pearson""",1.493473,"""mock_loader_1""","""A""",0,"""mock_predictor_1"""
"""spearman""",-1.151092,"""mock_loader_1""","""A""",0,"""mock_predictor_1"""
"""pearson""",0.181622,"""mock_loader_1""","""A""",0,"""mock_predictor_1"""
"""spearman""",-1.079719,"""mock_loader_1""","""A""",0,"""mock_predictor_1"""
"""pearson""",-0.471091,"""mock_loader_2""","""A""",0,"""mock_predictor_1"""
…,…,…,…,…,…
"""spearman""",-1.441644,"""mock_loader_1""","""A""",1,"""mock_predictor_2"""
"""pearson""",-0.30614,"""mock_loader_2""","""A""",1,"""mock_predictor_2"""
"""spearman""",0.032192,"""mock_loader_2""","""A""",1,"""mock_predictor_2"""


In [149]:
sns.boxplot(df, y="pearson_score", x="model_name")

ValueError: Could not interpret value `pearson_score` for `y`. An entry with this name does not appear in `data`.